# Aprendizado de Máquina — Aula prática 07

## Classificação e Classificadores Gaussianos

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

Em todo o Bloco I, a resposta $Y$ era um número, e a qualidade de uma predição se
media pela distância até o valor verdadeiro. A partir desta aula, $Y$ passa a ser uma
**classe**: um tumor é benigno ou maligno, uma mensagem é spam ou não é. Não existe
mais errar por pouco. O classificador acerta a classe ou erra.

Boa parte do que construímos continua valendo. Na regressão, o melhor preditor
possível era a média condicional $E[Y \mid X=x]$. Na classificação, o melhor
classificador possível também depende de uma quantidade condicional, a probabilidade
$P(Y=1 \mid X=x)$: se ela fosse conhecida, bastaria responder, em cada ponto $x$, a
classe mais provável. Como ela não é conhecida, cada método desta aula é uma maneira
diferente de estimá-la.

Vamos começar por uma população que nós mesmos inventamos. A vantagem é que, nela,
sabemos quanto vale $P(Y=1 \mid X=x)$ em qualquer ponto. Isso permite calcular o
melhor classificador possível e medir a que distância dele cada método fica. Só na
última seção passamos a um banco de dados real, onde não existe essa referência.

### Objetivos

Ao final deste notebook, você deve ser capaz de:

- calcular o classificador de Bayes e o erro dele numa população conhecida;
- ajustar o Bayes ingênuo, o LDA, o QDA e a regressão logística, e explicar a forma
  da fronteira de cada um a partir do que o método supõe;
- dizer como `predict_proba` e `predict` se relacionam;
- explicar, com uma conta de parâmetros, quando o LDA ganha do QDA e quando perde;
- lembrar que a `LogisticRegression` do `scikit-learn` já vem regularizada, e saber o
  que o parâmetro `C` controla.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

Além dos pacotes de sempre, precisamos dos quatro classificadores da aula: o
`GaussianNB` (Bayes ingênuo), o `LinearDiscriminantAnalysis` (LDA), o
`QuadraticDiscriminantAnalysis` (QDA) e a `LogisticRegression`, que mora no mesmo
módulo das regressões lineares da Aula 02. Os dois nomes longos ganham os apelidos
`LDA` e `QDA`.

Também importamos a `multivariate_normal`, do `scipy`, que calcula a densidade de uma
normal multivariada. A população deste notebook é feita de duas normais que nós
escolhemos, e é com essa função que vamos calcular as probabilidades verdadeiras.

In [ ]:
import sklearn.linear_model as skl
import sklearn.model_selection as skm
from sklearn.discriminant_analysis import (LinearDiscriminantAnalysis as LDA,
                                           QuadraticDiscriminantAnalysis as QDA)
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from scipy.stats import multivariate_normal

In [ ]:
import warnings
warnings.filterwarnings("ignore")

---
## 2. Uma população conhecida, e o melhor classificador possível

A população tem duas classes do mesmo tamanho. Dentro de cada classe, o vetor
$X = (X_1, X_2)$ segue uma normal bivariada, com média e matriz de covariância
próprias:

$$X \mid Y=0 \sim N\left(\begin{bmatrix}0\\0\end{bmatrix},
   \begin{bmatrix}1{,}00 & 0{,}75\\ 0{,}75 & 1{,}00\end{bmatrix}\right),
  \qquad
  X \mid Y=1 \sim N\left(\begin{bmatrix}2{,}20\\0{,}77\end{bmatrix},
   \begin{bmatrix}1{,}60 & -0{,}85\\ -0{,}85 & 0{,}70\end{bmatrix}\right).$$

Vale reparar nos números fora da diagonal. Na classe 0 eles são positivos: $X_1$ e
$X_2$ tendem a crescer juntas. Na classe 1 são negativos: quando uma cresce, a outra
tende a diminuir. As duas nuvens de pontos apontam, portanto, em direções diferentes,
e essa diferença vai aparecer várias vezes ao longo do notebook.

Como conhecemos as duas densidades, $f_0$ e $f_1$, e as proporções das classes,
$\pi_0 = \pi_1 = 1/2$, o Teorema de Bayes dá a probabilidade de cada classe em
qualquer ponto:

$$P(Y=1 \mid X=x) = \frac{\pi_1 f_1(x)}{\pi_0 f_0(x) + \pi_1 f_1(x)}.$$

O **classificador de Bayes** responde a classe mais provável, isto é, responde $1$
quando essa probabilidade passa de $1/2$. As notas mostram que nenhum classificador
tem risco menor que o dele.

A célula abaixo põe essa população em código. A função `amostra` sorteia $n$
observações, metade de cada classe, e a função `prob_bayes` calcula a probabilidade
verdadeira. As duas aceitam uma covariância diferente para a classe 1, o que vai ser
útil na Seção 4. Sorteamos um treino de 400 observações e um teste de 40 mil; o teste
é grande para que os erros medidos nele quase não variem de um sorteio para outro.

In [ ]:
S0 = np.array([[1.00, 0.75], [0.75, 1.00]])
S1 = np.array([[1.60, -0.85], [-0.85, 0.70]])
mu0, mu1 = np.array([0.0, 0.0]), np.array([2.2, 2.2 * 0.35])
pi0 = pi1 = 0.5


def amostra(n, rng, S1=S1):
    n0 = n // 2
    X = np.vstack([rng.multivariate_normal(mu0, S0, size=n0),
                   rng.multivariate_normal(mu1, S1, size=n - n0)])
    y = np.r_[np.zeros(n0), np.ones(n - n0)].astype(int)
    return X, y


def prob_bayes(X, S1=S1):
    """P(Y=1 | X=x), exata -- so existe porque inventamos a populacao."""
    f0 = pi0 * multivariate_normal(mu0, S0).pdf(X)
    f1 = pi1 * multivariate_normal(mu1, S1).pdf(X)
    return f1 / (f0 + f1)


rng = np.random.default_rng(31)
X, y = amostra(400, rng)
X_te, y_te = amostra(40_000, np.random.default_rng(99))

erro_bayes = np.mean((prob_bayes(X_te) > 0.5).astype(int) != y_te)
print(f"erro do classificador de Bayes: {erro_bayes:.4f}")
print(f"acuracia maxima possivel      : {1 - erro_bayes:.4f}")
print(f"erro de quem chuta sempre a mesma classe: {min(y_te.mean(), 1 - y_te.mean()):.4f}")

O classificador de Bayes erra $9{,}65\%$ das observações de teste. E ele não estimou
nada: usou as densidades verdadeiras. O erro vem só da sobreposição das duas nuvens.
Onde as duas classes aparecem misturadas, não há como acertar sempre, e o melhor que
se pode fazer é apostar na classe mais provável.

Esse número faz, na classificação, o papel que o $\sigma^2$ fazia na regressão. É um
piso: nenhum dos métodos que vamos ajustar tem risco menor, por mais dados que
receba. No outro extremo, quem chuta sempre a mesma classe erra metade das vezes,
porque as classes têm o mesmo tamanho.

Antes de ajustar qualquer modelo, vale olhar os dados. A figura mostra as 400
observações de treino e a fronteira do classificador de Bayes, que é a curva em que
$P(Y=1 \mid X=x) = 1/2$.

In [ ]:
xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - .8, X[:, 0].max() + .8, 320),
                     np.linspace(X[:, 1].min() - .8, X[:, 1].max() + .8, 320))
grade = np.c_[xx.ravel(), yy.ravel()]

fig, ax = subplots(figsize=(4.8, 3.6))
ax.scatter(X[y == 0, 0], X[y == 0, 1], s=8, color="steelblue", alpha=0.6,
           label="classe 0")
ax.scatter(X[y == 1, 0], X[y == 1, 1], s=8, color="crimson", alpha=0.6,
           label="classe 1")
ax.contour(xx, yy, prob_bayes(grade).reshape(xx.shape), levels=[0.5],
           colors="green", linewidths=1.6, linestyles="--")
ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")
ax.set_title("treino (n=400) e a fronteira de Bayes (tracejada)", fontsize=9)
ax.legend(fontsize=8);

A fronteira de Bayes é uma hipérbole, uma curva de dois ramos. O ramo de baixo passa
entre as duas nuvens e se dobra em volta da nuvem azul. O ramo de cima, à direita,
atravessa uma região quase sem dados e separa um pedaço do plano que o classificador
de Bayes atribui à classe 0. O motivo está no formato das nuvens: a azul é alongada
na diagonal, na direção em que $x_1$ e $x_2$ crescem juntas, e a vermelha é estreita
nessa direção. Bem longe das médias, subindo pela diagonal, a densidade da classe 0
cai mais devagar que a da classe 1, e a classe 0 volta a ser a mais provável. Na
próxima figura, o fundo colorido de cada painel mostra que classe o método atribui a
cada região.

É esse desenho que os quatro métodos da aula vão tentar reproduzir a partir das 400
observações, sem conhecer as densidades.

---
## 3. Os quatro classificadores da aula

Na prática, $P(Y=1 \mid X=x)$ é desconhecida. A saída é estimá-la com os dados de
treino, colocar a estimativa no lugar da probabilidade verdadeira e responder a classe
mais provável. As notas chamam isso de classificador *plug-in*. Os quatro métodos da
aula seguem essa receita e diferem no que supõem para chegar à estimativa:

- o **Bayes ingênuo** (`GaussianNB`) supõe que, dentro de cada classe, as coordenadas
  de $X$ são independentes, cada uma com distribuição normal;
- o **LDA** supõe que cada classe é normal e que as duas classes têm a **mesma**
  matriz de covariância;
- o **QDA** também supõe que cada classe é normal, mas deixa cada uma ter a **sua
  própria** matriz de covariância;
- a **regressão logística** não supõe nada sobre a distribuição de $X$. Supõe apenas
  que o logaritmo da razão de chances, $\log \frac{P(Y=1 \mid x)}{P(Y=0 \mid x)}$, é
  uma função linear de $x$.

Os três primeiros são *generativos*: descrevem como os dados de cada classe são
gerados e chegam a $P(Y=1 \mid x)$ pelo Teorema de Bayes, como fizemos acima com as
densidades verdadeiras. A logística é *discriminativa*: modela $P(Y=1 \mid x)$
diretamente.

Sabendo como a população foi gerada, qual dos quatro você espera que se saia melhor?
Pense um pouco antes de rodar a célula. Ela ajusta os quatro no mesmo treino e desenha
a fronteira de cada um (linha preta) sobre a fronteira de Bayes (tracejada verde). O
desenho fica numa função, `desenhar_fronteiras`, porque vamos repeti-lo na Seção 4.

In [ ]:
modelos = [("Bayes ingenuo", GaussianNB()),
           ("LDA (cov. comum)", LDA()),
           ("QDA (cov. por classe)", QDA()),
           ("Logistica", skl.LogisticRegression())]


def desenhar_fronteiras(X, y, prob_verdadeira, modelos):
    # ajusta cada modelo e desenha a fronteira dele (preta) sobre a de Bayes (verde)
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - .8, X[:, 0].max() + .8, 320),
                         np.linspace(X[:, 1].min() - .8, X[:, 1].max() + .8, 320))
    grade = np.c_[xx.ravel(), yy.ravel()]
    P_bayes = prob_verdadeira(grade).reshape(xx.shape)

    fig, axes = subplots(1, len(modelos), figsize=(11, 3.0), sharex=True, sharey=True)
    for ax, (nome, m) in zip(axes, modelos):
        m.fit(X, y)
        P = m.predict_proba(grade)[:, 1].reshape(xx.shape)
        ax.contourf(xx, yy, P, levels=[0, 0.5, 1], colors=["steelblue", "crimson"],
                    alpha=0.10)
        ax.contour(xx, yy, P_bayes, levels=[0.5], colors="green", linewidths=1.6,
                   linestyles="--")
        ax.contour(xx, yy, P, levels=[0.5], colors="black", linewidths=1.3)
        ax.scatter(X[y == 0, 0], X[y == 0, 1], s=6, color="steelblue", alpha=0.6)
        ax.scatter(X[y == 1, 0], X[y == 1, 1], s=6, color="crimson", alpha=0.6)
        ax.set_title(nome, fontsize=9); ax.set_xticks([]); ax.set_yticks([])


desenhar_fronteiras(X, y, prob_bayes, modelos)

A figura já sugere quem foi melhor, mas vale medir. O método `score` de um
classificador devolve a **acurácia**, que é a proporção de acertos; o erro é um menos
ela. A função abaixo calcula o erro de cada modelo nas 40 mil observações de teste e o
**excesso** sobre o classificador de Bayes, que é quanto cada método perde por não
conhecer as densidades.

In [ ]:
def erros_no_teste(modelos, X, y, X_te, y_te, erro_bayes):
    linhas = []
    for nome, m in modelos:
        erro = 1 - m.fit(X, y).score(X_te, y_te)
        linhas.append({"classificador": nome, "erro no teste": erro,
                       "excesso sobre Bayes": erro - erro_bayes})
    linhas.append({"classificador": "Bayes (inatingivel)", "erro no teste": erro_bayes,
                   "excesso sobre Bayes": 0.0})
    return pd.DataFrame(linhas).set_index("classificador").round(4)


erros_no_teste(modelos, X, y, X_te, y_te, erro_bayes)

O QDA fica a apenas $0{,}0017$ do piso. Faz sentido: geramos os dados exatamente como
o QDA supõe, com duas normais de covariâncias diferentes. Quando a suposição de um
método generativo coincide com a verdade, estimar os parâmetros e colocá-los na regra
de Bayes funciona muito bem, e 400 observações já bastam para isso. Repare, na figura,
que a fronteira do QDA acompanha a de Bayes até no ramo de cima, numa região quase sem
dados. Ali quem desenha a fronteira não são as observações, é a suposição de
normalidade, e ela é verdadeira nesta população.

Os outros três perdem entre $2{,}6$ e $3{,}5$ pontos percentuais, e a figura mostra de
onde vem cada perda. O LDA e a logística traçam retas, e nenhuma reta acompanha uma
fronteira curva. O Bayes ingênuo consegue curvar a fronteira, mas não do jeito certo,
porque trata as duas coordenadas como independentes dentro de cada classe, e aqui elas
não são.

Um detalhe: a logística erra um pouco menos que o LDA ($0{,}1230$ contra $0{,}1263$),
embora os dois tracem retas. Para escolher a sua reta, o LDA se apoia nas próprias
suposições, a normalidade e a covariância comum, e a segunda é falsa nesta população.
A logística supõe apenas que a fronteira é uma reta e a estima diretamente,
maximizando a verossimilhança de $Y$ dado $x$, sem passar por covariância nenhuma.

---
## 4. O que cada método fez com as covariâncias

Os três métodos generativos estimam covariâncias, e dá para abri-los e comparar o que
estimaram com as matrizes verdadeiras. O LDA e o QDA guardam as matrizes estimadas
quando criados com `store_covariance=True`. O Bayes ingênuo guarda só a variância de
cada coordenada, no atributo `var_`; a covariância que ele usa sem dizer é a matriz
diagonal com essas variâncias.

In [ ]:
qda_aj = QDA(store_covariance=True).fit(X, y)
lda_aj = LDA(store_covariance=True).fit(X, y)
nb_aj = GaussianNB().fit(X, y)


def correlacao(S):
    return S[0, 1] / np.sqrt(S[0, 0] * S[1, 1])


for k, S in [(0, S0), (1, S1)]:
    print(f"classe {k}, covariancia VERDADEIRA:\n", S)
    print(f"classe {k}, estimada pelo QDA:\n", qda_aj.covariance_[k].round(3), "\n")
print("estimada pelo LDA (a MESMA para as duas classes):\n", lda_aj.covariance_.round(3))
print("\nusada pelo Bayes ingenuo na classe 0 (so a diagonal):\n",
      np.diag(nb_aj.var_[0]).round(3), "\n")

for k, S in [(0, S0), (1, S1)]:
    print(f"correlacao na classe {k}: verdadeira {correlacao(S):+.3f}, "
          f"estimada pelo QDA {correlacao(qda_aj.covariance_[k]):+.3f}")
print("correlacao que o Bayes ingenuo assume: 0 nas duas classes")

Comece pelo QDA. As duas matrizes estimadas ficam perto das verdadeiras, e cada uma
com o sinal certo fora da diagonal: positivo na classe 0 e negativo na classe 1. As
correlações estimadas, $+0{,}79$ e $-0{,}83$, estão perto das verdadeiras, $+0{,}75$ e
$-0{,}80$. É por isso que a fronteira do QDA acompanhou a de Bayes.

O LDA precisa devolver uma única matriz para as duas classes. Ele junta as dispersões
de cada classe em torno da própria média, e o resultado não descreve bem nenhuma das
duas: o termo fora da diagonal, $-0{,}19$, fica entre o $+0{,}75$ da classe 0 e o
$-0{,}85$ da classe 1. Com uma matriz comum às duas classes, os termos quadráticos em
$x$ se cancelam quando comparamos as classes, e por isso a fronteira do LDA é uma
reta.

O Bayes ingênuo acerta as variâncias, na diagonal, mas trata as covariâncias como
zero. Nas duas classes isso é falso, e daí vem a fronteira curva, mas desalinhada, do
primeiro painel da figura.

Toda a comparação até aqui foi feita numa população em que o QDA estava certo. Para
ver o outro lado, refazemos a figura e a tabela numa população em que o LDA está
certo: a mesma de antes, mas com a classe 1 usando a covariância da classe 0,
$\Sigma_1 = \Sigma_0$. As duas nuvens passam a ter o mesmo formato e a diferir só na
posição. As sementes são as da Seção 2, para que a única mudança seja a covariância.

In [ ]:
Xi, yi = amostra(400, np.random.default_rng(31), S1=S0)
Xi_te, yi_te = amostra(40_000, np.random.default_rng(99), S1=S0)
erro_bayes_i = np.mean((prob_bayes(Xi_te, S1=S0) > 0.5).astype(int) != yi_te)
print(f"erro de Bayes com covariancias iguais: {erro_bayes_i:.4f}")
print(f"com covariancias diferentes, era     : {erro_bayes:.4f}")

desenhar_fronteiras(Xi, yi, lambda Z: prob_bayes(Z, S1=S0), modelos)
erros_no_teste(modelos, Xi, yi, Xi_te, yi_te, erro_bayes_i)

Com covariâncias iguais, a fronteira de Bayes vira uma reta e o LDA passa a ser o
modelo certo. A tabela confirma: o excesso dele é de $-0{,}0001$. Um excesso negativo
não quer dizer que o LDA seja melhor que o classificador de Bayes, cujo risco é o menor
possível; quer dizer que a diferença entre os dois é menor que a precisão de uma medida
feita com 40 mil pontos de teste.

O QDA quase não piora: $+0{,}0015$. O modelo do QDA contém o do LDA como caso
particular, aquele em que as duas covariâncias são iguais, e com 400 observações as
duas matrizes estimadas saem parecidas. Na figura, ele traça praticamente a mesma reta
que o LDA onde há dados, e só se curva de verdade num canto vazio, no alto à esquerda.
O que ele paga é a variância de estimar seis números de covariância em vez de três, com
os mesmos dados.

A logística também está certa agora, porque com covariâncias iguais o logaritmo da
razão de chances é exatamente linear em $x$. Ela fica em $+0{,}0032$, um pouco atrás do
LDA, e a razão é de eficiência: quando as normais são de fato a verdade, o LDA usa essa
informação e estima a reta com menos variância do que a logística, que não a usa.

O Bayes ingênuo, ao contrário, piora bastante: o excesso vai de $+0{,}0355$ para
$+0{,}0737$. Igualar as covariâncias não resolve o problema dele, porque agora as duas
classes têm correlação $0{,}75$ entre as coordenadas, e ele continua supondo zero. Com
covariância comum $\Sigma$, a fronteira de Bayes é perpendicular ao vetor
$\Sigma^{-1}(\mu_1 - \mu_0)$, que leva a correlação em conta. Ignorando-a, o Bayes
ingênuo fica, em essência, com a direção de $\mu_1 - \mu_0$ (aqui as variâncias são
iguais a 1), e as duas direções formam um ângulo de quase 48 graus.

---
## 5. Probabilidades e decisões: `predict_proba` e `predict`

Os classificadores do `scikit-learn` que estimam probabilidades têm dois métodos
parecidos. O `predict_proba` devolve as probabilidades estimadas, com uma coluna por
classe. O `predict` devolve uma classe. Como um se relaciona com o outro?

Com duas classes, `predict` responde $1$ quando a probabilidade estimada da classe 1
passa de $0{,}5$. A célula confere isso no QDA e, como conhecemos a probabilidade
verdadeira, também compara as estimativas com ela.

In [ ]:
qda_aj = QDA().fit(X, y)
p = qda_aj.predict_proba(X_te)[:, 1]

print("predict == (predict_proba > 0.5)?",
      np.array_equal(qda_aj.predict(X_te), (p > 0.5).astype(int)))
print(f"\nprobabilidade estimada, 5 primeiras: {np.round(p[:5], 3)}")
print(f"probabilidade VERDADEIRA, as mesmas : {np.round(prob_bayes(X_te[:5]), 3)}")

fig, ax = subplots(figsize=(5.2, 3.0))
ax.scatter(prob_bayes(X_te[:4000]), p[:4000], s=3, alpha=0.25)
ax.plot([0, 1], [0, 1], color="black", lw=1)
ax.set_xlabel("P(Y=1|x) verdadeira"); ax.set_ylabel("estimada pelo QDA")
ax.set_title("probabilidade estimada x verdadeira (QDA, n=400)", fontsize=9);

A comparação confirma que `predict` é o `predict_proba` seguido de um corte em
$0{,}5$.

No gráfico, as estimativas do QDA seguem as probabilidades verdadeiras de perto, mas
não se espalham ao acaso em volta da diagonal: a maioria dos pontos forma uma faixa
estreita um pouco abaixo dela, e um grupo menor fica acima. É que o erro de estimação
não é um ruído independente em cada ponto. O QDA estimou médias e covariâncias um pouco
diferentes das verdadeiras, e isso deforma a função de probabilidade inteira de um
jeito suave, puxando as estimativas para baixo numa parte do plano e para cima em
outra.

O quinto ponto mostra o efeito que essa deformação pode ter: a probabilidade
verdadeira é $0{,}529$ e a estimada é $0{,}485$. Os dois números caem em lados opostos
do corte, e nesse ponto o QDA responde 0 onde o classificador de Bayes responderia 1.

O corte em $0{,}5$ minimiza a probabilidade de erro, e é a escolha certa quando os dois
tipos de erro custam o mesmo. Em muitos problemas não custam: deixar passar um tumor
maligno é muito mais grave do que pedir um exame a mais para um benigno. Nesses casos o
corte deve ser outro. As probabilidades estimadas continuam as mesmas; muda só o
número com que as comparamos. Esse é um dos assuntos da Aula 08.

---
## 6. LDA ou QDA?

Na Seção 3 o QDA ganhou com folga, e na população de covariâncias iguais quase empatou
com o LDA. Pode parecer, então, que não há motivo para usar o LDA, já que o QDA é mais
geral. O motivo aparece quando há poucos dados, e passa por uma conta de parâmetros.

Com duas classes em $\mathbb{R}^p$, os dois métodos estimam dois vetores de médias, com
$2p$ números. O LDA estima uma matriz de covariância, que tem $p(p+1)/2$ entradas
livres por ser simétrica; o QDA estima uma por classe, o dobro. A célula faz a conta
para três valores de $p$, sem contar as proporções das classes. (No código, o número de
covariáveis se chama `d`, porque o nome `p` já está guardando as probabilidades da
Seção 5.)

In [ ]:
for d in (2, 10, 50):
    p_lda = d * (d + 1) // 2 + 2 * d
    p_qda = 2 * (d * (d + 1) // 2) + 2 * d
    print(f"d = {d:3d}: LDA estima {p_lda:5d} parametros, QDA estima {p_qda:5d}"
          f"   (razao {p_qda/p_lda:.2f}x)")

Em $p=2$, o QDA estima só três parâmetros a mais que o LDA. Em $p=10$ já são 55 a
mais, e em $p=50$, $1\,275$. Mais parâmetros significam mais variância, como vimos na
Aula 01, e daí vem uma regra prática conhecida: com poucos dados, prefira o LDA,
mesmo que a suposição de covariância comum seja falsa.

Vamos testar essa regra em $p=10$. A população tem duas classes normais com
covariâncias diferentes, de modo que o QDA é o modelo certo, e médias que diferem só
nas três primeiras coordenadas. Para cada tamanho de treino $n$, sorteamos 120
amostras, ajustamos os dois métodos em cada uma e medimos a acurácia em 20 mil
observações novas. É a mesma simulação, com a mesma semente, da figura das notas que
compara LDA e QDA, e os números que vão aparecer são os da figura.

A célula abaixo só define as ferramentas: a população em $\mathbb{R}^p$ (as duas
covariâncias são sorteadas uma vez e ficam fixas), a função que repete a comparação e a
que desenha o resultado.

In [ ]:
_CACHE_Q = {}


def cov_fixa(d, qual):
    # covariancia da classe `qual` em dimensao d: sempre a mesma matriz
    if (d, qual) not in _CACHE_Q:
        g = np.random.default_rng(1000 + qual)
        M = g.normal(size=(d, d))
        _CACHE_Q[(d, qual)] = (M @ M.T) / d + np.eye(d) * 0.5
    return _CACHE_Q[(d, qual)]


def gaussianas_dim(n, d, rng):
    n0 = n // 2
    mu = np.zeros(d)
    sinal = [1.6, 1.0, 0.7]                    # so as 3 primeiras coordenadas separam
    mu[:min(3, d)] = sinal[:min(3, d)]
    X = np.vstack([rng.multivariate_normal(np.zeros(d), cov_fixa(d, 0), size=n0),
                   rng.multivariate_normal(mu, cov_fixa(d, 1), size=n - n0)])
    return X, np.r_[np.zeros(n0), np.ones(n - n0)].astype(int)


def comparar_lda_qda(sortear, ns, n_rep, semente=32):
    # acuracia media de LDA e QDA em funcao de n, para um dado gerador de amostras
    rng_c = np.random.default_rng(semente)
    Xt, yt = sortear(20_000, np.random.default_rng(99))
    saida = {"LDA": np.zeros((n_rep, len(ns))), "QDA": np.zeros((n_rep, len(ns)))}
    for b in range(n_rep):
        for j, n in enumerate(ns):
            Xb, yb = sortear(int(n), rng_c)
            for nome, M in [("LDA", LDA), ("QDA", QDA)]:
                try:
                    saida[nome][b, j] = M().fit(Xb, yb).score(Xt, yt)
                except np.linalg.LinAlgError:     # covariancia singular
                    saida[nome][b, j] = np.nan
    return pd.DataFrame({"LDA": np.nanmean(saida["LDA"], axis=0),
                         "QDA": np.nanmean(saida["QDA"], axis=0)},
                        index=pd.Index(ns, name="n"))


def grafico(res, titulo):
    fig, ax = subplots(figsize=(5.2, 3.1))
    ax.plot(res.index, res["LDA"], "o-", ms=4, color="steelblue", label="LDA")
    ax.plot(res.index, res["QDA"], "s-", ms=4, color="crimson", label="QDA")
    ax.set_xscale("log"); ax.set_xticks(res.index)
    ax.set_xticklabels(res.index, fontsize=7); ax.minorticks_off()
    ax.set_xlabel("n (treino)"); ax.set_ylabel("acuracia em 20 mil observacoes novas")
    ax.set_title(titulo, fontsize=9); ax.legend(fontsize=8)

Com as funções prontas, rodamos a simulação em $p=10$. São 120 amostras para cada um
dos oito tamanhos de treino, e a célula leva alguns segundos.

In [ ]:
res10 = comparar_lda_qda(lambda n, r: gaussianas_dim(n, 10, r),
                         np.array([30, 50, 80, 150, 300, 700, 1500, 4000]), n_rep=120)
grafico(res10, "d = 10, covariancias diferentes por classe")
res10.round(4)

Com $n=30$, o LDA acerta $74{,}8\%$ e o QDA, $69{,}2\%$: o modelo errado ganha do
modelo certo por mais de cinco pontos percentuais. São 15 observações por classe para
estimar, em cada classe, uma matriz com 55 números, e as estimativas saem tão
instáveis que a flexibilidade do QDA atrapalha mais do que ajuda. Em $n=50$ a situação
já se inverte, e a vantagem do QDA cresce com $n$, até quase sete pontos em $n=4000$,
quando há dados suficientes para estimar bem as duas covariâncias.

É a troca entre viés e variância da Aula 01, agora num problema de classificação. O
LDA tem viés, porque supõe uma covariância comum que não existe, mas estima menos
coisas. O QDA não tem esse viés, mas precisa de dados para pagar pela flexibilidade.

A simulação começou em $n=30$ por um motivo. Com $n=20$ são só dez observações por
classe em $\mathbb{R}^{10}$. Depois de subtraída a média da classe, os dez vetores
somam zero e, por isso, ficam num subespaço de dimensão no máximo nove. A covariância
amostral deles tem então posto no máximo nove e não é invertível. Como a densidade
normal usa a inversa da covariância, o QDA não tem como ser calculado. Veja o que
acontece quando tentamos:

In [ ]:
X20, y20 = gaussianas_dim(20, 10, np.random.default_rng(7))
try:
    QDA().fit(X20, y20)
    print("o QDA foi ajustado sobre uma covariancia singular: versoes antigas do "
          "scikit-learn so emitem um aviso, e os avisos estao silenciados")
except np.linalg.LinAlgError as erro:
    print("o scikit-learn recusou o ajuste:\n")
    print(erro)

A mensagem diz exatamente isso: cada classe tem 10 observações para 10 covariáveis, e
o `scikit-learn` exige mais observações do que covariáveis em cada classe. O LDA não
tem esse problema com $n=20$, porque junta as dispersões das duas classes numa matriz
só, estimada com as 20 observações. Com tão poucos dados por classe, o QDA falha antes
mesmo de chegarmos a medir a acurácia.

Agora a mesma comparação em $p=2$, com a população da Seção 2. Nela o QDA estima só
três parâmetros a mais que o LDA. Se a regra prática valesse sem ressalvas, o LDA
deveria ganhar ao menos quando $n$ é bem pequeno. Desta vez incluímos $n=20$, que em
duas dimensões não causa problema nenhum.

In [ ]:
res2 = comparar_lda_qda(amostra, np.array([20, 30, 50, 80, 150, 300, 700, 1500, 4000]),
                        n_rep=60)
grafico(res2, "d = 2, a populacao da Secao 2")
res2.round(4)

Em $p=2$ o QDA ganha em toda a faixa, já a partir de $n=20$ ($88{,}2\%$ contra
$85{,}4\%$). Nesta população não existe um regime de poucos dados em que compense
impor uma covariância comum.

A diferença entre as duas situações vem de dois fatores, e a regra prática só fala de
um deles. O primeiro é o que o QDA tem a perder: os parâmetros que ele estima a mais,
$p(p+1)/2$, que são 3 em $p=2$ e 55 em $p=10$. Vinte observações dão conta de três
parâmetros extras sem dificuldade. O segundo é o que o QDA tem a ganhar, e isso depende
de quanto as covariâncias das classes diferem de verdade. Na população da Seção 2 elas
diferem muito, com correlações de sinais opostos, e o viés do LDA é grande. Numa
população em que as covariâncias fossem parecidas, o QDA teria pouco a ganhar, por mais
barato que fosse.

Uma versão mais completa da regra tem, então, duas partes: o LDA tende a ganhar quando
$n$ é pequeno em comparação com os parâmetros extras do QDA, que crescem com $p^2$; e o
QDA só compensa quando as covariâncias das classes são de fato diferentes. Num problema
real não sabemos nenhuma das duas coisas de antemão, e quem decide é a validação
cruzada.

---
## 7. A `LogisticRegression` já vem regularizada

Quem aprendeu regressão logística num curso de estatística espera que
`LogisticRegression()` devolva o estimador de máxima verossimilhança. Não é o que
acontece: por padrão, o `scikit-learn` acrescenta uma penalidade $\ell_2$ sobre os
coeficientes, a mesma ideia da Ridge da Aula 02.

A intensidade da penalidade é controlada pelo parâmetro `C`, que é o **inverso** dela:
um `C` pequeno significa penalidade forte, ao contrário do `alpha` da Ridge e do
Lasso. O padrão é `C=1`. Para obter o ajuste sem penalidade, usa-se `C=np.inf`.

A célula ajusta a logística com vários valores de `C`, num problema com dez
covariáveis (a população da Seção 6, com $n=200$), e imprime o tamanho do vetor de
coeficientes e quantos deles são diferentes de zero.

In [ ]:
X_g, y_g = gaussianas_dim(200, 10, np.random.default_rng(5))

print(f"{'C':>10} {'||beta||':>10} {'nao nulos':>10}")
for c in [0.001, 0.01, 0.1, 1.0, 100.0]:
    m = skl.LogisticRegression(C=c, max_iter=5000).fit(X_g, y_g)
    print(f"{c:10.3f} {np.linalg.norm(m.coef_):10.3f} {int((np.abs(m.coef_) > 1e-6).sum()):10d}")

sem_pen = skl.LogisticRegression(C=np.inf, max_iter=5000).fit(X_g, y_g)
print(f"{'sem penal.':>10} {np.linalg.norm(sem_pen.coef_):10.3f}")

Com `C=1`, o padrão, a norma dos coeficientes é $2{,}198$, contra $2{,}448$ do ajuste
sem penalidade: uma redução de cerca de 10% que acontece sem ninguém pedir. Com
$C=0{,}001$ o vetor quase desaparece. Repare também que os dez coeficientes continuam
diferentes de zero em todas as linhas. Como na Ridge, a penalidade $\ell_2$ encolhe os
coeficientes, mas não zera nenhum.

Isso tem duas consequências práticas. Se você for interpretar ou relatar os
coeficientes, precisa saber que eles vêm de um ajuste penalizado. E, se for usar a
penalidade, `C` é um hiperparâmetro como outro qualquer: escolha-o por validação
cruzada, dentro de um `Pipeline` com `StandardScaler`, porque a penalidade soma
coeficientes de colunas diferentes e por isso depende da escala delas (Aula 06).

---
## 8. Um caso real: diagnóstico de câncer de mama

Para terminar, deixamos a população inventada. O conjunto `breast_cancer`, que vem com
o `scikit-learn`, descreve 569 tumores por 30 medidas feitas em imagens de células
(raio, textura, perímetro, área e outras), e cada tumor está classificado como benigno
ou maligno. Aqui não conhecemos $P(Y=1 \mid x)$, então não há erro de Bayes para
servir de referência. O que dá para fazer é comparar os métodos entre si.

Separamos 30% dos tumores para teste, mantendo nos dois conjuntos a proporção de cada
classe (`stratify`).

In [ ]:
from sklearn.datasets import load_breast_cancer

dados = load_breast_cancer()
Xr, yr = dados.data, dados.target
print(f"{Xr.shape[0]} tumores, {Xr.shape[1]} medidas")
print(f"classes: {dict(zip(dados.target_names, np.bincount(yr)))}")
print(f"prevalencia da classe 1 (benigno): {yr.mean():.3f}")

X_tr, X_ts, y_tr, y_ts = skm.train_test_split(Xr, yr, test_size=0.3,
                                              random_state=0, stratify=yr)

Antes de comparar os métodos, um problema prático. Várias das 30 medidas são quase
redundantes; raio, perímetro e área de um mesmo tumor, por exemplo, dizem praticamente
a mesma coisa. Com colunas assim, a covariância estimada dentro de uma classe fica
muito perto de ser singular, e o QDA das versões recentes do `scikit-learn` se recusa a
ajustar:

In [ ]:
try:
    QDA().fit(X_tr, y_tr)
    print("o QDA foi ajustado")
except np.linalg.LinAlgError as erro:
    print("o scikit-learn recusou o ajuste:\n")
    print(erro)

A mensagem sugere o remédio: o parâmetro `reg_param`, que puxa a covariância estimada
de cada classe levemente na direção da matriz identidade. Isso afasta a matriz da
singularidade sem mudar muito o modelo. Vamos usar `reg_param=1e-4`.

Na comparação abaixo, a logística entra num `Pipeline` com `StandardScaler` e tem o `C`
escolhido por validação cruzada, como recomendado na seção anterior. Os três métodos
generativos recebem as colunas na escala original. Para cada método medimos a acurácia
de duas formas: por validação cruzada com 5 dobras dentro do treino e no conjunto de
teste. No caso da logística, a busca pelo `C` é refeita dentro de cada dobra, o que faz
dessa validação cruzada uma validação aninhada (Aula 03). A última linha é o
classificador que chuta sempre a classe mais frequente.

In [ ]:
candidatos = {
    "Bayes ingenuo": GaussianNB(),
    "LDA": LDA(),
    "QDA (reg_param=1e-4)": QDA(reg_param=1e-4),
    "Logistica (C por CV)": skm.GridSearchCV(
        Pipeline([("sc", StandardScaler()),
                  ("clf", skl.LogisticRegression(max_iter=5000))]),
        {"clf__C": np.logspace(-3, 3, 13)}, cv=5, scoring="accuracy"),
}

linhas = []
for nome, m in candidatos.items():
    m.fit(X_tr, y_tr)
    cv = skm.cross_val_score(m, X_tr, y_tr, cv=5, scoring="accuracy").mean()
    linhas.append({"classificador": nome, "acuracia (CV no treino)": cv,
                   "acuracia (teste)": m.score(X_ts, y_ts)})
linhas.append({"classificador": "chutar a classe mais frequente",
               "acuracia (CV no treino)": np.nan,
               "acuracia (teste)": max(y_ts.mean(), 1 - y_ts.mean())})
pd.DataFrame(linhas).set_index("classificador").round(4)

Os quatro métodos passam de 92% de acurácia no teste, muito acima dos $62{,}6\%$ de
quem responde sempre "benigno". A logística é a melhor nas duas colunas, com $0{,}9799$
na validação cruzada e $0{,}9591$ no teste.

Entre o LDA e o QDA, as duas colunas discordam. No teste o LDA fica à frente por
$0{,}0117$; na validação cruzada é o QDA que fica à frente, por $0{,}0025$. Não há
contradição: as duas diferenças são pequenas e foram medidas com poucos dados. O teste
tem 171 tumores, e $0{,}0117$ de 171 são dois tumores. Com essa quantidade de dados, o
mais honesto é dizer que LDA e QDA empataram. A conta de parâmetros da Seção 6 está
presente, e não é pequena: o QDA estima $30 \times 31/2 = 465$ números de covariância
para cada classe, e a classe dos malignos tem só 148 tumores no treino. Mesmo assim, o
sinal deste problema é forte o bastante para que ela não decida a disputa.

O Bayes ingênuo fica atrás nas duas colunas: cerca de um ponto abaixo do LDA na
validação cruzada e três no teste. As 30 medidas são muito correlacionadas entre si, e
supor que elas são independentes dentro de cada classe tem um custo.

Por fim, repare no que a tabela não mostra. Todos os números são acurácias, e a
acurácia trata como iguais dois erros muito diferentes: chamar de benigno um tumor
maligno e chamar de maligno um tumor benigno. Em diagnóstico médico esses dois erros
não se equivalem de jeito nenhum, e a tabela não diz quantos de cada tipo cada modelo
cometeu. Separar os tipos de erro é o ponto de partida da Aula 08.

---
## Resumo

| Seção | O que fizemos | O que vimos |
|---|---|---|
| 2 | calculamos o classificador de Bayes numa população conhecida | ele erra $9{,}65\%$, e nenhum método tem risco menor que esse |
| 3 | ajustamos os quatro métodos e desenhamos as fronteiras | o QDA, cuja suposição é a verdade aqui, fica a $0{,}0017$ do piso; LDA e logística traçam retas e perdem cerca de três pontos |
| 4 | comparamos as covariâncias estimadas com as verdadeiras | o LDA impõe uma matriz comum e o Bayes ingênuo zera as correlações; com covariâncias iguais, o LDA vira o modelo certo e o Bayes ingênuo piora |
| 5 | comparamos `predict_proba` e `predict` | `predict` é a probabilidade estimada com um corte em $0{,}5$, e o corte é uma escolha |
| 6 | medimos LDA contra QDA em função de $n$ | em $p=10$ o LDA ganha com poucos dados e o QDA nem ajusta com $n=20$; em $p=2$ o QDA ganha sempre |
| 7 | variamos o `C` da `LogisticRegression` | o padrão `C=1` já penaliza e encolhe os coeficientes cerca de 10%, sem zerar nenhum |
| 8 | comparamos os quatro no `breast_cancer` | a logística vai melhor, LDA e QDA empatam dentro do ruído, e a acurácia não diz que tipo de erro cada um comete |

**Leitura recomendada.** [AME] §7.1 (classificação, risco 0–1 e o classificador de
Bayes) e §8.1.2–8.1.4 (a logística, o Bayes ingênuo e a análise discriminante).
[ISLP] §2.2.3 (o classificador de Bayes e o erro de Bayes), §4.3 (a logística) e §4.4
(LDA, QDA e Bayes ingênuo; a Figura 4.9 deles se parece com as nossas figuras das
Seções 3 e 4, e o §4.4.3 traz a conta de parâmetros da Seção 6).

**Para praticar.** `Lista teorica 07.pdf` (teórica, com gabarito) e
`Lista prática 07.ipynb` (prática, para completar as lacunas), nesta mesma pasta.

**A seguir.** Na Aula 08 vamos separar os tipos de erro que a acurácia mistura e ver
como escolher o corte quando um erro custa mais do que o outro.